# Model training with CONNIE image dataset

In [1]:
%run ./../notebook_init.py

import os
import torch
import optuna
import mlflow

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from collections import Counter
from itertools import product
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import Subset
from sklearn.metrics import classification_report, confusion_matrix

from core import DATA_FOLDER, RESULTS_FOLDER

from scripts.connie_training_utils import ModelTraining, TransformedSubset, \
    Seed, get_test_transform, IMG_SIZE, get_train_transform,\
    resnet18_model, NPYFolderDataset

Load file paths and set the computation device to GPU if available; otherwise, use CPU, and initialize the random seed

In [2]:
#train_data = os.path.join(DATA_FOLDER, "train_data_png_full")
train_data = os.path.join(DATA_FOLDER, "train_data_npy_full")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
seed = Seed()

cuda:0


In [3]:
trainval_dataset = NPYFolderDataset(train_data)
trainval_set = Subset(trainval_dataset, list(range(len(trainval_dataset))))

Compute dataset mean and standard deviation, then define training and test transforms with normalization

In [4]:
#basic_transform = transforms.Compose([
#    transforms.Resize(IMG_SIZE),
#    transforms.ToTensor()
#])
#
## Load dataset without transform
#full_dataset_transform = datasets.ImageFolder(train_data,
#                                              transform=basic_transform)
#
#mean, std = calculate_mean_std(full_dataset_transform)
#test_transform = get_test_transform(mean, std)
#train_transform = get_train_transform(mean, std)

Split train + validation and test set

In [5]:
#from torchvision import datasets

## Original dataset
#trainval_dataset = datasets.ImageFolder(train_data)

## Classes to remove
#classes_to_remove = ["Alpha"]
#classes_to_remove = []

#
#keep_classes = [c for c in trainval_dataset.classes if c not in classes_to_remove]
#
#old_to_new = {trainval_dataset.class_to_idx[c]: i for i, c in enumerate(keep_classes)}
#
#indices_to_keep = [
#    i for i, (_, label) in enumerate(trainval_dataset.samples) if label in old_to_new
#]
#
#trainval_dataset_remapped = datasets.ImageFolder(train_data)
#trainval_dataset_remapped.samples = [
#    (trainval_dataset.samples[i][0], old_to_new[trainval_dataset.samples[i][1]])
#    for i in indices_to_keep
#]
#trainval_dataset_remapped.targets = [label for _, label in trainval_dataset_remapped.samples]
#trainval_dataset_remapped.classes = keep_classes
#trainval_dataset_remapped.class_to_idx = {c: i for i, c in enumerate(keep_classes)}
#
## Now create the Subset for compatibility
#trainval_set = Subset(trainval_dataset_remapped, list(range(len(trainval_dataset_remapped.samples))))


## Training with K-fold

In [6]:
# Define your parameter grid
param_grid = {
    "learning_rate": [5e-4],
    'weight_decay': [5e-5],
    "step_size": [10],
    "gamma": [0.5]
}

# Create all combinations
grid = list(product(
    param_grid["learning_rate"],
    param_grid["weight_decay"],
    param_grid["step_size"],
    param_grid["gamma"]
))

k_folds = 5
num_epochs = 100

class_idx_map = trainval_dataset.class_to_idx
print("Classes index:", class_idx_map)

Classes index: {'Blob': 0, 'Diffusion Hit': 1, 'Electron': 2, 'Muon': 3, 'Others': 4}


## Hyperparameters Tuning

In [7]:
from pathlib import Path
mlflow.set_tracking_uri(Path(DATA_FOLDER) / "mlruns")
#mlflow.set_tracking_uri(os.path.join(DATA_FOLDER, "mlruns"))

def objective_resnet18(trial):
    k_folds = 5
    num_epochs = 100
    model_training = ModelTraining()

    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    wd = trial.suggest_float("wd", 1e-5, 1e-2, log=True)
    step_size = trial.suggest_int("step", 5, 50)
    gamma = trial.suggest_float("gamma", 0.1, 0.9)

    hyperparam = {"lr": lr, "wd": wd, "step": step_size, "gamma": gamma}

    with mlflow.start_run(nested=True, run_name=f"ResNet_trial_{trial.number}"):
        mlflow.set_tag("model_type", "ResNet-18")
        mlflow.set_tag("kfold_splits", k_folds)
        mlflow.log_params(hyperparam)
        
        metrics = model_training.train_model_kfold_multiclass(
            device=device,
            dataset=trainval_set,
            num_epochs=num_epochs,
            k_folds=k_folds,
            seed=seed,
            model=resnet18_model,
            hyperparam=hyperparam
        )

        mlflow.log_metrics({
            "mean_train_accuracy": metrics["mean_train_accuracy"],
            "std_train_accuracy": metrics["std_train_accuracy"],
            "mean_train_loss": metrics["mean_train_loss"],
        
            "mean_val_accuracy": metrics["mean_val_accuracy"],
            "std_val_accuracy": metrics["std_val_accuracy"],
            "mean_val_loss": metrics["mean_val_loss"],
            "std_val_loss": metrics["std_val_loss"],
        
            "mean_precision": metrics["mean_precision"],
            "std_precision": metrics["std_precision"],
            "mean_recall": metrics["mean_recall"],
            "std_recall": metrics["std_recall"],
            "mean_f1_macro": metrics["mean_f1_macro"],
            "std_f1_macro": metrics["std_f1_macro"]
        })
        mlflow.log_dict(metrics, "full_metrics.json")

        if "all_val_true" in metrics and "all_val_preds" in metrics:
            all_val_true = metrics["all_val_true"]
            all_val_preds = metrics["all_val_preds"]

            metrics_dir = os.path.join(RESULTS_FOLDER,
                                       "metrics_resnet18_multiclass_no_alpha")
            os.makedirs(metrics_dir, exist_ok=True)

            class_names = trainval_set.dataset.classes

            unique_labels = np.unique(np.concatenate([all_val_true, all_val_preds]))
            label_mapping = {old: new for new, old in enumerate(sorted(unique_labels))}
            
            all_val_true_mapped = np.array([label_mapping[y] for y in all_val_true])
            all_val_preds_mapped = np.array([label_mapping[y] for y in all_val_preds])

            # Classification report
            report = classification_report(
                all_val_true_mapped,
                all_val_preds_mapped,
                target_names=class_names,
                output_dict=True,
                zero_division=0
            )
            report_df = pd.DataFrame(report).transpose()

            report_path = os.path.join(metrics_dir,
                                       f"trial_{trial.number}_classification_report.csv")
            report_df.to_csv(report_path)
            mlflow.log_artifact(report_path)

            # === Confusion matrix ===
            cm = confusion_matrix(all_val_true_mapped,
                                  all_val_preds_mapped)

            fig, ax = plt.subplots(figsize=(8, 6))
            sns.heatmap(
                cm,
                annot=True,
                fmt="d",
                cmap="Blues",
                xticklabels=class_names,
                yticklabels=class_names,
                ax=ax
            )
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            fig.tight_layout()

            cm_path = os.path.join(metrics_dir,
                                   f"trial_{trial.number}_confusion_matrix.png")
            fig.savefig(cm_path)
            mlflow.log_artifact(cm_path)
            plt.close(fig)

        return metrics["mean_f1_macro"]


In [8]:
class_idx_map = trainval_set.dataset.class_to_idx

   
mlflow.set_experiment(f"Tuning_Resnet18_multiclass_no_alpha_fixed_2")
study_resnet18 = optuna.create_study(direction="maximize")
study_resnet18.optimize(lambda trial: objective_resnet18(trial),
                        n_trials=100)

print(f"Best trials for Resnet-18:")
for i, t in enumerate(study_resnet18.best_trials):
    print(f"Trial #{t.number}")
    print(f"  Values (Val Accuracy, Val Loss): {t.values}")
    print(f"  Params: ")
    for key, value in t.params.items():
        print(f"    {key}: {value}")

2026/04/08 14:43:47 INFO mlflow.tracking.fluent: Experiment with name 'Tuning_Resnet18_multiclass_no_alpha_fixed_2' does not exist. Creating a new experiment.
[I 2026-04-08 14:43:47,696] A new study created in memory with name: no-name-68a343ec-fed3-43f3-bddf-df16320f0563



==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 1.1983 Acc: 0.5407
Val Loss: 2.0337 Acc: 0.2514
Val Precision: 0.3815 Recall: 0.4145 F1: 0.1416

Epoch 2/100 — Fold 1
----------
Train Loss: 0.7406 Acc: 0.7623
Val Loss: 0.7682 Acc: 0.6997
Val Precision: 0.5045 Recall: 0.7191 F1: 0.5507

Epoch 3/100 — Fold 1
----------
Train Loss: 0.7146 Acc: 0.7602
Val Loss: 1.2917 Acc: 0.5978
Val Precision: 0.5541 Recall: 0.6890 F1: 0.5736

Epoch 4/100 — Fold 1
----------
Train Loss: 0.6165 Acc: 0.7473
Val Loss: 0.5408 Acc: 0.7807
Val Precision: 0.6497 Recall: 0.7592 F1: 0.6852

Epoch 5/100 — Fold 1
----------
Train Loss: 0.5823 Acc: 0.7815
Val Loss: 0.9511 Acc: 0.6341
Val Precision: 0.4288 Recall: 0.4978 F1: 0.4347

Epoch 6/100 — Fold 1
----------
Train Loss: 0.7183 Acc: 0.7669
Val Loss: 0.8055 Acc: 0.7207
Val Precision: 0.5620 Recall: 0.7346 F1: 0.6091

Epoch 7/100 — Fold 1
----------
Train Loss: 0.5452 Acc: 0.8158
Val Loss: 0.5430 Acc: 0.8003
Val Preci

[I 2026-04-08 15:03:11,807] Trial 0 finished with value: 0.7810987435966424 and parameters: {'lr': 0.002152229676776595, 'wd': 2.4861089206157938e-05, 'step': 31, 'gamma': 0.2694325531281221}. Best is trial 0 with value: 0.7810987435966424.



==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 0.9182 Acc: 0.6508
Val Loss: 2.3307 Acc: 0.3729
Val Precision: 0.3990 Recall: 0.4347 F1: 0.2354

Epoch 2/100 — Fold 1
----------
Train Loss: 0.6235 Acc: 0.7850
Val Loss: 0.7850 Acc: 0.6830
Val Precision: 0.5226 Recall: 0.7822 F1: 0.5830

Epoch 3/100 — Fold 1
----------
Train Loss: 0.5441 Acc: 0.8046
Val Loss: 0.5840 Acc: 0.8142
Val Precision: 0.6640 Recall: 0.8466 F1: 0.7272

Epoch 4/100 — Fold 1
----------
Train Loss: 0.5164 Acc: 0.8130
Val Loss: 0.4945 Acc: 0.8184
Val Precision: 0.6907 Recall: 0.8424 F1: 0.7479

Epoch 5/100 — Fold 1
----------
Train Loss: 0.4780 Acc: 0.8420
Val Loss: 0.4582 Acc: 0.8547
Val Precision: 0.7220 Recall: 0.8098 F1: 0.7475

Epoch 6/100 — Fold 1
----------
Train Loss: 0.5356 Acc: 0.8217
Val Loss: 0.3929 Acc: 0.8687
Val Precision: 0.7493 Recall: 0.8881 F1: 0.8029

Epoch 7/100 — Fold 1
----------
Train Loss: 0.4547 Acc: 0.8528
Val Loss: 0.3962 Acc: 0.8813
Val Preci

[I 2026-04-08 15:22:05,728] Trial 1 finished with value: 0.8610637247125249 and parameters: {'lr': 0.00046070957047126757, 'wd': 0.007363161963362294, 'step': 10, 'gamma': 0.1848172388355284}. Best is trial 1 with value: 0.8610637247125249.


...


==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 1.1878 Acc: 0.6159
Val Loss: 2.4803 Acc: 0.2947
Val Precision: 0.2651 Recall: 0.4796 F1: 0.2327

Epoch 2/100 — Fold 1
----------
Train Loss: 0.6870 Acc: 0.7739
Val Loss: 0.7325 Acc: 0.7081
Val Precision: 0.4965 Recall: 0.7302 F1: 0.5571

Epoch 3/100 — Fold 1
----------
Train Loss: 0.6913 Acc: 0.7176
Val Loss: 1.0867 Acc: 0.6075
Val Precision: 0.5778 Recall: 0.7238 F1: 0.6016

Epoch 4/100 — Fold 1
----------
Train Loss: 0.6080 Acc: 0.7641
Val Loss: 0.7072 Acc: 0.7291
Val Precision: 0.6394 Recall: 0.7821 F1: 0.6637

Epoch 5/100 — Fold 1
----------
Train Loss: 0.5798 Acc: 0.8029
Val Loss: 1.0555 Acc: 0.6564
Val Precision: 0.3828 Recall: 0.5758 F1: 0.3989

Epoch 6/100 — Fold 1
----------
Train Loss: 0.6336 Acc: 0.7753
Val Loss: 0.4035 Acc: 0.8547
Val Precision: 0.6686 Recall: 0.7814 F1: 0.6935

Epoch 7/100 — Fold 1
----------
Train Loss: 0.5497 Acc: 0.8025
Val Loss: 0.4315 Acc: 0.8492
Val Preci

[I 2026-04-10 02:17:58,493] Trial 98 finished with value: 0.8676561421251998 and parameters: {'lr': 0.0011861227127860936, 'wd': 0.00021214689847626908, 'step': 17, 'gamma': 0.2210056737286328}. Best is trial 74 with value: 0.873948061422602.



==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 1.2161 Acc: 0.5708
Val Loss: 2.6156 Acc: 0.0545
Val Precision: 0.0436 Recall: 0.2825 F1: 0.0367

Epoch 2/100 — Fold 1
----------
Train Loss: 0.8459 Acc: 0.6631
Val Loss: 0.8847 Acc: 0.6955
Val Precision: 0.4561 Recall: 0.7035 F1: 0.5051

Epoch 3/100 — Fold 1
----------
Train Loss: 0.6578 Acc: 0.7427
Val Loss: 0.6107 Acc: 0.7807
Val Precision: 0.5188 Recall: 0.6788 F1: 0.5450

Epoch 4/100 — Fold 1
----------
Train Loss: 0.6610 Acc: 0.7379
Val Loss: 0.8689 Acc: 0.5698
Val Precision: 0.5560 Recall: 0.6851 F1: 0.5354

Epoch 5/100 — Fold 1
----------
Train Loss: 0.5711 Acc: 0.7934
Val Loss: 1.1486 Acc: 0.6089
Val Precision: 0.4380 Recall: 0.5571 F1: 0.4452

Epoch 6/100 — Fold 1
----------
Train Loss: 0.6112 Acc: 0.7700
Val Loss: 0.6987 Acc: 0.7416
Val Precision: 0.6280 Recall: 0.7850 F1: 0.6611

Epoch 7/100 — Fold 1
----------
Train Loss: 0.6329 Acc: 0.7875
Val Loss: 0.5502 Acc: 0.8184
Val Preci

[I 2026-04-10 02:52:39,964] Trial 99 finished with value: 0.8427665236425728 and parameters: {'lr': 0.0015211393104004958, 'wd': 0.00046453526171437567, 'step': 17, 'gamma': 0.2171212954690256}. Best is trial 74 with value: 0.873948061422602.


Best trials for Resnet-18:
Trial #74
  Values (Val Accuracy, Val Loss): [0.873948061422602]
  Params: 
    lr: 0.0004481479831586827
    wd: 0.0001716842332539542
    step: 24
    gamma: 0.4402853450898717
